In [4]:
import pandas as pd
import numpy as np

job_postings_fact_2023 = pd.read_csv(r'C:\Users\mechz\Documents\Jobs_Analysis\csv_files_2023\job_postings_fact.csv')
skills_dim_2023 = pd.read_csv(r'C:\Users\mechz\Documents\Jobs_Analysis\csv_files_2023\skills_dim.csv')
skills_job_dim_2023 = pd.read_csv(r'C:\Users\mechz\Documents\Jobs_Analysis\csv_files_2023\skills_job_dim.csv')
company_dim_2023 = pd.read_csv(r'C:\Users\mechz\Documents\Jobs_Analysis\csv_files_2023\company_dim.csv')

job_postings_fact_2024 = pd.read_csv(r'C:\Users\mechz\Documents\Jobs_Analysis\csv_files_2024\job_postings_fact.csv')
skills_dim_2024 = pd.read_csv(r'C:\Users\mechz\Documents\Jobs_Analysis\csv_files_2024\skills_dim.csv')
skills_job_dim_2024 = pd.read_csv(r'C:\Users\mechz\Documents\Jobs_Analysis\csv_files_2024\skills_job_dim.csv')
company_dim_2024 = pd.read_csv(r'C:\Users\mechz\Documents\Jobs_Analysis\csv_files_2024\company_dim.csv')

job_postings_fact_2023 = job_postings_fact_2023[job_postings_fact_2023['salary_year_avg'].notna()]
job_postings_fact_2023 = job_postings_fact_2023.drop(columns=['job_title', 'job_location', 'search_location', 'salary_rate', 'salary_hour_avg', 'company_id'])
job_postings_fact_2023['job_via'] = job_postings_fact_2023['job_via'].str.replace('via ', '')

temporary_df_1 = pd.merge(left=job_postings_fact_2023, right=skills_job_dim_2023, how='inner', on='job_id')
temporary_df_1 = pd.merge(left=temporary_df_1, right=skills_dim_2023, how='inner', on='skill_id')

job_postings_fact_2024 = job_postings_fact_2024[job_postings_fact_2024['salary_year_avg'].notna()]
job_postings_fact_2024 = job_postings_fact_2024.drop(columns=['job_title', 'job_location', 'search_location', 'salary_rate', 'salary_hour_avg', 'company_id'])
job_postings_fact_2024['job_via'] = job_postings_fact_2024['job_via'].str.replace('via ', '')

temporary_df_2 = pd.merge(left=job_postings_fact_2024, right=skills_job_dim_2024, how='inner', on='job_id')
temporary_df_2 = pd.merge(left=temporary_df_2, right=skills_dim_2024, how='inner', on='skill_id')

main_df = pd.concat([temporary_df_1, temporary_df_2])

list_of_vias = main_df.groupby('job_via').count().sort_values(by='job_id', ascending=False).head(10).index
main_df.loc[main_df['job_via'].isin(list_of_vias) == False, 'job_via'] = 'other'
list_of_skills = main_df.groupby('skills').count().sort_values(by='job_id', ascending=False).head(30).index
main_df.loc[main_df['skills'].isin(list_of_skills) == False, 'skills'] = 'other'
list_of_countries = main_df.groupby('job_country').count().sort_values(by='job_id', ascending=False).head(10).index
main_df.loc[main_df['job_country'].isin(list_of_countries) == False, 'job_country'] = 'other'


main_df['job_posted_date'] = pd.to_datetime(main_df['job_posted_date'])
main_df['job_posted_year'] = main_df['job_posted_date'].dt.year
main_df['job_posted_month_sine'] = np.sin(2 * np.pi * main_df['job_posted_date'].dt.month / 12)

main_df['job_schedule_type'] = main_df['job_schedule_type'].str.replace(' ', '')
main_df['job_schedule_type'] = main_df['job_schedule_type'].str.replace('and', ',')
main_df['job_schedule_type'] = main_df['job_schedule_type'].str.replace(',,', ',')
schedule_dummies = main_df['job_schedule_type'].str.get_dummies(sep=',')
main_df = pd.concat([main_df, schedule_dummies], axis=1).drop(columns=['job_schedule_type'])

main_df = pd.get_dummies(main_df, columns=['skills'])
non_skill_columns = [column for column in main_df.columns if not column.startswith('skills_')]
skill_columns = [column for column in main_df.columns if column.startswith('skills_')]
agg_dict = {column:'first' for column in non_skill_columns if column != 'job_id'}
agg_dict.update({column: 'max' for column in skill_columns})
main_df = main_df.groupby('job_id').agg(agg_dict)
main_df = pd.get_dummies(main_df, columns=['job_title_short', 'job_via', 'job_country']).drop(columns=['skill_id', 'type', 'job_posted_date']).reset_index(drop=True)
main_df = main_df[main_df['Perdiem'] == 0]
main_df = main_df.drop(columns=['Perdiem'])

main_df.astype(float).to_csv('job_postings.csv', index=False)